# Problem Set 3: Exploring multimodal embeddings with AstroCLIP 
AstroCLIP paper: https://arxiv.org/abs/2310.03024. Code adapted from Francois Lanusse & Liam Parker by Mariel Pettee.

**Software requirements:** All requirements are installable via `pip` or `uv`. Note that I have included a `requirements.txt` file to facilitate package installation as well as a `pyproject.toml` file if you prefer using `uv` (I recommend it!). You will need to use Python 3.10. 

**Datasets:**
Available via HuggingFace (see code below).

**Grading:**
This problem set will be graded as a quiz within Canvas. Note that for reproducibility purposes, I recommend using `Kernel` $\rightarrow$ `Restart Kernel and Run All Cells...` before submitting your answers.

**Deadline:** 
The Canvas quiz will close by end-of-day on Wednesday, November 26th, 2025.

In [ ]:
import torch
import datasets
from datasets import load_dataset
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from astroclip.data.datamodule import AstroClipCollator
import numpy as np

### WARNING: don't change this cell -- setting these seeds will ensure that your answers are consistent with mine 
seed = 1234 
np.random.seed(seed)
torch.random.manual_seed(seed);

# Load the data

In [ ]:
# Loads cross-matched data between DESI and Legacy Survey images
dset = load_dataset('EiffL/AstroCLIP', streaming=True, split='train')
dset = dset.with_format('torch')

# Creates a torch data loader for the data
dloader = DataLoader(dset, batch_size=64, collate_fn=AstroClipCollator(), drop_last=True)
iter_dset = iter(dloader)

In [ ]:
# We can extract one batch of objects like so:
batch = next(iter_dset)

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 1: Batch keys</span>
Which of the following variables are present in each batch?

- 'image'
- 'index'
- 'redshift'
- 'galaxy'
- 'spectra'
- 'z'
- 'spectrum'
- 'rgb'
- 'targetid'

Now let's plot some galaxy images:

In [ ]:
images = batch['image']

# Assuming images is a tensor of shape (batch_size, channels, height, width)
# and we want to plot the first 64 images in an 8x8 grid.
fig, axes = plt.subplots(8, 8, figsize=(10, 10))
axes = axes.flatten()

for i in range(min(len(images), 64)):
    img = images[i].permute(1, 2, 0).numpy()  # Convert to HWC for matplotlib
    axes[i].imshow(img)
    axes[i].axis('off')

plt.tight_layout()

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 2: Batch #1 redshift</span>
What is the average redshift of the first batch of 64 galaxies? Report your answer with precision `1e-2`.

# Compute embeddings with AstroCLIP

Download the AstroCLIP model weights:

In [ ]:
!mkdir -p ../data/pset_3/
!wget -nc -O ../data/pset_3/astroclip.ckpt "https://huggingface.co/polymathic-ai/astroclip/resolve/main/astroclip.ckpt?download=true"

In [ ]:
from astroclip.models import AstroClipModel

# Loads the model from downloaded checkpoint
model = AstroClipModel.load_from_checkpoint(checkpoint_path = "../data/pset_3/astroclip.ckpt",).eval()

In [ ]:
with torch.no_grad():
  # Apply the model on images
  embeddings = model(batch['image'], input_type='image') # AstroCLIP understands 'image' or 'spectum' as input_type

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 3: Embedding dimensionality</span>
What is the dimensionality of the embedding for each individual galaxy? Report your answer as a single integer.

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 4: Visualizing the data</span>
Run the code below to generate an interactive visualization of the 64 galaxies in this batch, organized into graph-like structure(s) using a $k$-nearest neighbor clustering algorithm. (It might take a minute or two to load.) Note that you can click and drag on the galaxies to tease apart their graph structures. 

How many distinct subgraphs are present? (Note that you are welcome to use another method, e.g. using the `networkx` package, to determine this -- the visualization is cool but not essential.)

In [ ]:
%pylab inline
import pyarrow
from tqdm import tqdm
import networkx as nx
from pyvis.network import Network
from IPython.display import HTML
import base64
from io import BytesIO
import matplotlib.pyplot as plt
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import normalize
import torch
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact
from IPython.display import display


def graph_from_embeddings(embeddings, images, k=5, symmetrize=True):
    """
    Builds a k-NN graph adjacency matrix using scikit-learn (cosine distance).

    Returns a sparse adjacency matrix (numpy array or CSR).
    """
    # Convert to NumPy and normalize (cosine similarity = dot of L2-normalized vectors)
    emb_np = embeddings.cpu().numpy()

    # scikit-learn uses cosine *distance*, so we negate similarity in behavior
    A = kneighbors_graph(emb_np, n_neighbors=k, metric='cosine', mode='connectivity', include_self=False)

    if symmetrize:
        A = A.maximum(A.T)  # make it symmetric (undirected)

    G = nx.Graph()
    A = A.tocoo()

    for i in range(A.shape[0]):
        G.add_node(i, image_tensor=images[i].permute(1, 2, 0).cpu().numpy() if images is not None else None)

    for i, j, w in zip(A.row, A.col, A.data):
        G.add_edge(i, j, weight=w)

    return G

def encode_image_base64(image_array, scale=1.):
    fig = plt.figure(figsize=(scale, scale), dpi=100)
    plt.axis("off")
    plt.imshow(image_array)
    buf = BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)
    img_base64 = base64.b64encode(buf.read()).decode('utf-8')
    return f"data:image/png;base64,{img_base64}"

def draw_interactive_graph_colab(graph):
    net = Network(height='750px', width='100%', bgcolor='#000000', font_color='white', notebook=False, cdn_resources='remote')

    # Encode each image as base64 and use as node icon
    for node_id, data in graph.nodes(data=True):
        img_url = encode_image_base64(data['image_tensor'])
        net.add_node(
            int(node_id),
            shape="image",
            image=img_url,
            title=f"Galaxy {node_id}",
            size=50
        )

    for src, tgt, data in graph.edges(data=True):
        net.add_edge(int(src), int(tgt), value=float(data['weight']) * 5,
                     color='rgba(255, 255, 255, 0.5)')

    # Generate the HTML string (without writing to file)
    html_str = net.generate_html()

    # Display directly in Colab
    return HTML(html_str)

In [ ]:
G = graph_from_embeddings(embeddings, images=images, k=2)
draw_interactive_graph_colab(G)

Now we will grab a few more batches to create a (slightly) larger database to query:

In [ ]:
embeddings = []
redshift = []
images = []

for i in tqdm(range(3)):
    batch = next(iter_dset)
    with torch.no_grad():
        emb = model(batch['image'],input_type='image')

    images.append(batch['image'].numpy())
    redshift.append(batch['redshift'].numpy())
    embeddings.append(emb.numpy())

embeddings = np.concatenate(embeddings)
redshift = np.concatenate(redshift)
images = np.concatenate(images)

In [ ]:
print(embeddings.shape)

## 🔎 Similarity Search in Embedding Space

In this section, we perform a **similarity search** to find galaxies that are most similar to a selected query image, based on their positions in the learned **embedding space**.

We use **cosine similarity** to measure how close each image embedding is to the embedding of the selected query image. This allows us to identify objects that share similar high-level visual or morphological features, as captured by the embedding model.

The steps are:
- Select a query galaxy image.
- Compute cosine similarity between its embedding and all others in the dataset.
- Retrieve and display the **top 16 most similar images** (excluding the query itself).

This technique enables **fast, content-based retrieval** from large image datasets, and can be a powerful tool for exploratory data analysis and serendipitous discovery.

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 5: Similarity search</span>
Run the code below to perform a similarity search for a galaxy image. Select **image #6** as your query image -- the galaxy at the center should be blue-ish and round.

What is the index of the most similar image to image #6 (not counting the original image)? 

In [ ]:
query_images = images[:10]
selected_index = {'value': 0}  # Use a mutable dict

# Define the display function
def show_query_image(index):
    selected_index['value'] = index  # Store the selection
    img = query_images[index]
    # Convert CHW to HWC for matplotlib
    img_hwc = np.transpose(img, (1, 2, 0))

    plt.figure(figsize=(3, 3))
    plt.imshow(img_hwc)
    plt.axis('off')
    plt.title(f'Selected Query Image: {index}')
    plt.show()

# Use interact to create a dropdown or slider
interact(show_query_image, index=(0, len(query_images) - 1));

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query_index = ???

similarity = cosine_similarity(embeddings[query_index].reshape(1, -1),
                               embeddings).squeeze()

## 📈 Redshift Estimation with k-Nearest Neighbors (k-NN) Regression

In this section, we use a **k-Nearest Neighbors (k-NN) regressor** to predict galaxy redshifts directly from their image embeddings.

The idea is simple: for a given embedding, find its `k` closest neighbors in the embedding space and return the **average redshift** of those neighbors. This method relies on the assumption that similar embeddings (i.e., similar galaxies) should have similar redshifts.

Here’s what we do:
- Split the dataset into training and testing sets.
- Train a `KNeighborsRegressor` using the training embeddings and redshifts.
- Predict redshifts for the test set.
- Visualize the performance by plotting **true vs. predicted redshifts**.

This provides a quick baseline and helps assess how well the learned embedding space captures redshift-related information.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Prepare data for k-NN regression
X_train, X_test, y_train, y_test = train_test_split(embeddings, redshift, test_size=0.2)

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 6: kNeighborsRegressor</span>

Train a kNN regressor with `n_neighbors = 10`, then evaluate it on your test set (defined in the cell above). Documentation here: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html

What is the MSE between `y_pred` and `y_test`? Report your answer with precision `1e-3` (rounding if necessary). 

In [ ]:
knn_regressor = ???

# Predict redshift on the test set
y_pred = ???

# Plot true vs predicted redshift (optional)
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("True Redshift")
plt.ylabel("Predicted Redshift")
plt.title(f"True vs. Predicted Redshift")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2) # Add a diagonal line
plt.show()

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 7: kNeighborsRegressor, part 2</span>

What is the $R^2$ score between `y_pred` and `y_test`? Report your answer with precision `1e-2` (rounding if necessary). 

## 🔍 Inferring Redshifts with Implicit Inference

In this section, we use Implicit Inference to model the conditional distribution of galaxy redshift given an image embedding. Specifically, we apply the **Sequential Neural Posterior Estimation (SNPE-A)** algorithm from the `sbi` library.

Rather than predicting a single redshift value, this approach learns the full **posterior distribution** ( $p(\text{redshift} \mid \text{embedding}) $), capturing uncertainty and potential ambiguities in the mapping from image features to physical properties.

Here's what we do:
- Treat our training set of embeddings (`X_train`) and corresponding redshifts (`y_train`) as observed simulations.
- Define a simple prior over redshift values (uniform between 0 and 1).
- Use SNPE-A to train a neural density estimator that approximates the posterior over redshift given a new embedding.

This allows us to move beyond point estimates and toward **probabilistic reasoning**, providing more informative outputs for downstream scientific analysis.


In [ ]:
from sbi.inference import SNPE_A
from sbi.utils import BoxUniform
from torch import Tensor

# Setting up the training data
x_obs = torch.tensor(X_train, dtype=torch.float32)
theta_obs = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1) # sbi expects parameter tensors to have shape (num_samples, num_parameters)

# Define a simple prior for the redshift. Assuming redshifts are between 0 and 1 for simplicity.
# In a real scenario, you'd define a prior that reflects your knowledge about the redshift distribution.
prior = ???

# Building the inference method
inference = ???(prior=prior)
inference.append_simulations(theta_obs, x_obs)

# Train the posterior estimator
# This step trains the neural network to learn p(theta | x) based on the provided data.
print("Training the SBI posterior estimator...")
density_estimator = inference.train()
print("Training complete.")

In [ ]:
from sbi.analysis import pairplot

# Now we can use the trained density estimator to estimate the posterior distribution of redshift
# for new observations (embeddings) in the test set X_test.

# Convert X_test to tensor
x_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# Estimate the posterior distribution for the first test sample
# You can loop through the test set or process in batches
print("\nEstimating posterior for a test sample...")
# Let's pick the first sample from the test set
test_sample_embedding = x_test_tensor[0].unsqueeze(0) # Add batch dimension

# Get the posterior object for this observation
posterior = inference.build_posterior(density_estimator=density_estimator)

# Sample from the posterior distribution for this observation
num_samples = 1000
posterior_samples = posterior.sample(sample_shape=(num_samples,), x=test_sample_embedding)

print(f"True redshift for this sample: {y_test[0]:.3f}")
# Calculate the mean of the posterior samples as a point estimate
predicted_redshift_mean = posterior_samples.mean().item()
print(f"Posterior mean redshift estimate: {predicted_redshift_mean:.3f}")

# Let's plot the posterior for the first test sample again
fig, axes = pairplot(posterior_samples, limits=[[0, 1]], labels=['Redshift'], figsize=(5,5))
plt.suptitle(f"Posterior distribution for Test Sample 0 (True Z: {y_test[0]:.3f})")

# Overlay the true redshift as a vertical line
true_z = y_test[0]
axes.axvline(true_z, color='red', linestyle='--', label=f'True Z = {true_z:.3f}')
axes.axvline(predicted_redshift_mean, color='blue', linestyle='--', label=f'Pred Z = {predicted_redshift_mean:.3f}')

axes.legend()

plt.show();

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 8: True redshift</span>

What is the true redshift for this sample? Report your answer with precision `1e-3` (rounding if necessary). 

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 9: SBI result with SNPE_A</span>

What is the average redshift from the posterior? Report your answer with precision `1e-3` (rounding if necessary). 

### <span style="border:3px; border-style:solid; padding: 0.15em; border-color: #1f77b4; color: #1f77b4;">Problem 10: SBI result with SNLE</span>

Now repeat the SBI analysis, but use SNLE instead of SNPE_A. 

What is the average redshift from the posterior? Report your answer with precision `1e-3` (rounding if necessary). 